# 3. Running Qudit-ADAPT

The loop, in four steps:

1. evaluate the gradient $g_j = i\langle\psi|[V_j, H_C]|\psi\rangle$ of every
   pool operator, at $\theta_j = 0$;
2. stop if $\|g\|_2 < \varepsilon$;
3. otherwise append the operator with the largest $|g_j|$;
4. re-optimize **all** parameters, warm-starting from
   $\boldsymbol\theta_k = (\boldsymbol\theta^*_{k-1}, 0)$.

We do this twice: live on a small instance so you can see the machinery run,
then on $G_1$ of the paper by loading the stored run — the real one takes
minutes, not seconds, and reproducing it is what
[`REPRODUCING.md`](../REPRODUCING.md) is for.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np

import time
from funciones.utilidades_bp import adapt_bp_scan

# Instancia chica: cinco vertices, cinco aristas. Toma ~15 s.
n_chico = 5
edges_chico = [(1, 2), (1, 3), (2, 4), (3, 5), (4, 5)]

t0 = time.time()
chico = adapt_bp_scan(n=n_chico, edges=edges_chico, l=1, epsilon=1e-2,
                      max_iteration=20, n_random=0, seed=0, show=False)
print(f"ran in {time.time() - t0:.1f} s")
print(f"pool         : {chico['pool_size']} operators")
print(f"converged at : {chico['num_ansatz_ops']} parameters ({chico['stop_reason']})")

E0c = chico["ground_energy"]
epsc = np.abs(np.asarray(chico["recycled_energy"]) - E0c) / abs(E0c)
print(f"eps_rel      : {epsc[0]:.3f} -> {epsc[-1]:.2e}")

ran in 12.0 s
pool         : 25 operators
converged at : 18 parameters (gradient_norm_below_epsilon)
eps_rel      : 0.333 -> 1.48e-02


## The warm start makes energy monotone

Because the new parameter enters at $\theta_{k+1}=0$, the enlarged ansatz
starts exactly where the previous one finished:

$$E_{k+1}(\boldsymbol\theta^*_k, 0) = E_k(\boldsymbol\theta^*_k).$$

So the energy can only go down. This is the *burrowing* mechanism of Grimsley
*et al.*: the previous local minimum becomes a saddle point in the enlarged
landscape, and there is always a descent direction to follow.

In [2]:
warm = np.asarray(chico["recycled_energy"])
saltos = np.diff(warm)
print(f"monotone non-increasing: {bool(np.all(saltos <= 1e-10))}")
print(f"largest uphill step    : {saltos.max():+.2e}   (should be <= 0)")

monotone non-increasing: True
largest uphill step    : -1.61e-04   (should be <= 0)


## Now the real instance

$G_1$ of the paper: six vertices, ten edges, $\ell=1$. This is the stored run
behind the red curve of Fig. 1a and the first row of Table I.

In [3]:
from funciones.utilidades_bp import load_bp_result

res = load_bp_result("fig1a_l1_curva.json")
E0 = res["ground_energy"]
eps = np.abs(np.asarray(res["recycled_energy"]) - E0) / abs(E0)

print(f"E_0 (exact)  = {E0:.10f}")
print(f"pool size    = {res['pool_size']}")
print(f"converged at = {res['num_ansatz_ops']} parameters ({res['stop_reason']})")
print(f"eps_rel      = {eps[-1]:.2e}")
print(f"r = |E/E_0|  = {abs(res['recycled_energy'][-1] / E0):.4f}"
      f"     (Table I reports 0.993)")

E_0 (exact)  = -20.0000000000
pool size    = 46
converged at = 21 parameters (gradient_norm_below_epsilon)
eps_rel      = 6.56e-03
r = |E/E_0|  = 0.9934     (Table I reports 0.993)


## Which operators it picked, and in what order

Careful with the ordering: the ansatz applies **right to left**, so
`ansatz_op_labels[0]` is the operator closest to the reference state — the
first one applied.

In [4]:
from funciones.utilidades_bp import conteo_compuertas

print(f"{'k':>3s}  {'operator':32s}{'sites':>6s}{'theta*':>10s}{'eps_rel':>11s}")
for k, (lbl, th) in enumerate(zip(res["ansatz_op_labels"], res["params"]), start=1):
    if 6 < k < res["num_ansatz_ops"] - 2:
        if k == 7:
            print(f"{'..':>3s}")
        continue
    print(f"{k:>3d}  {lbl:32s}{conteo_compuertas(lbl)['peso']:>6d}"
          f"{th:>10.4f}{eps[k]:>11.2e}")

  k  operator                         sites    theta*    eps_rel
  1  ((3, 'y'), (6, 'z'))                 2    1.5708   3.13e-01
  2  ((3, 'z'), (4, 'y'))                 2    1.5708   2.86e-01
  3  ((2, 'y'), (4, 'z'))                 2    1.5708   2.30e-01
  4  ((4, 'z'), (5, 'y'))                 2    1.4492   1.74e-01
  5  ((1, 'y'), (1, 'z'), (4, 'z'), (4, 'z'))     2    3.1416   5.29e-02
  6  ((3, 'y'), (4, 'z'))                 2    1.5708   4.60e-02
 ..
 19  ((3, 'y'), (3, 'z'))                 1    1.7899   1.31e-02
 20  ((3, 'y'), (4, 'z'))                 2    0.6690   7.60e-03
 21  ((1, 'z'), (1, 'z'), (3, 'y'), (3, 'z'))     2    1.3517   6.56e-03


## What the energy error hides

$\epsilon_{\rm rel} = 6.6\times10^{-3}$ reads as unconverged. But for a
combinatorial problem the observable of interest is not the energy — it is
whether *measuring* the state hands you the right colouring. And it already
does, well before the energy settles.

In [5]:
from funciones.utilidades_bp import reconstruir_estado_final, solucion_max3cut

psi = reconstruir_estado_final(res)
sol = solucion_max3cut(res, psi=psi)

print(f"most probable basis state : p = {sol['probabilidad']:.3f}")
print(f"edges it cuts             : {sol['num_cortadas']}/{res['num_edges']}"
      f"   (maximum possible: {sol['max_corte']})")
print(f"is that colouring optimal : {sol['es_optima']}")
print(f"monochromatic edges       : {sol['aristas_monocromaticas'] or 'none'}")
print(f"colouring                 : {sol['coloreo']}")

most probable basis state : p = 0.468
edges it cuts             : 10/10   (maximum possible: 10)
is that colouring optimal : True
monochromatic edges       : none
colouring                 : {1: 1, 2: 0, 3: 0, 4: 2, 5: 0, 6: 1}


So at 21 parameters the energy is still $6.6\times10^{-3}$ away, yet the single
most likely measurement outcome is already an exactly optimal colouring, with
probability $0.47$. That is a genuine difference from the molecular problems
ADAPT-VQE was designed for, where the energy itself is what you want.

**Next:** [`04_native_gate_count.ipynb`](04_native_gate_count.ipynb) — what
this ansatz costs on hardware.